In [0]:
%sql
select 
    *
from aic_data_fact.common_tv.tv_usage_monthly
where 1=1
    and platform_code like '%25%'
;

In [0]:
%sql
select 
    date_ym,platform_code,count(1)
from aic_data_fact.common_tv.tv_usage_monthly
where 1=1
    and platform_code like '%25%'
group by 
    date_ym,platform_code
order by 
    1,2

In [0]:
%sql
select 
    X_Device_Sales_Model
from aic_data_ods.tlamp.normal_log_webos25
where 1=1
    and date_ym='2026-05'
    

In [0]:
%sql
-- 인치 추출 로직: 처음 2자리 숫자 또는 'OLED' 뒤 2자리 숫자 추출
-- 추가적으로 QNED, NANO, MRGB, LX, UA 등 패턴도 확인
SELECT 
    X_Device_Sales_Model,
    -- 인치 추출: OLED 뒤 2자리, 처음 2자리, QNED/NANO/MRGB/LX/UA 뒤 2자리 등
    CASE 
        WHEN X_Device_Sales_Model LIKE 'OLED%' AND REGEXP_EXTRACT(X_Device_Sales_Model, 'OLED([0-9]{2})', 1) != '' 
            THEN REGEXP_EXTRACT(X_Device_Sales_Model, 'OLED([0-9]{2})', 1)
        WHEN REGEXP_EXTRACT(X_Device_Sales_Model, '^([0-9]{2})', 1) != '' 
            THEN REGEXP_EXTRACT(X_Device_Sales_Model, '^([0-9]{2})', 1)
        WHEN X_Device_Sales_Model LIKE 'QNED%' AND REGEXP_EXTRACT(X_Device_Sales_Model, 'QNED([0-9]{2})', 1) != '' 
            THEN REGEXP_EXTRACT(X_Device_Sales_Model, 'QNED([0-9]{2})', 1)
        WHEN X_Device_Sales_Model LIKE 'NANO%' AND REGEXP_EXTRACT(X_Device_Sales_Model, 'NANO([0-9]{2})', 1) != '' 
            THEN REGEXP_EXTRACT(X_Device_Sales_Model, 'NANO([0-9]{2})', 1)
        WHEN X_Device_Sales_Model LIKE 'MRGB%' AND REGEXP_EXTRACT(X_Device_Sales_Model, 'MRGB([0-9]{2})', 1) != '' 
            THEN REGEXP_EXTRACT(X_Device_Sales_Model, 'MRGB([0-9]{2})', 1)
        WHEN X_Device_Sales_Model LIKE 'LX%' AND REGEXP_EXTRACT(X_Device_Sales_Model, 'LX([0-9]{2})', 1) != '' 
            THEN REGEXP_EXTRACT(X_Device_Sales_Model, 'LX([0-9]{2})', 1)
        WHEN X_Device_Sales_Model LIKE 'UA%' AND REGEXP_EXTRACT(X_Device_Sales_Model, 'UA([0-9]{2})', 1) != '' 
            THEN REGEXP_EXTRACT(X_Device_Sales_Model, 'UA([0-9]{2})', 1)
        ELSE NULL
    END AS inch
FROM aic_data_ods.tlamp.normal_log_webos25
WHERE 1=1
    AND date_ym='2026-05'

In [0]:
%sql
SELECT platform_code, platform_version, count(1)
FROM aic_data_mart.tv_usage.hdmi_usage
WHERE 1=1
    AND date_ym='2026-05'
    AND platform_code like 'W2%'
GROUP BY platform_code, platform_version

In [0]:
%sql
SELECT 
    -- 인치 카테고리 컬럼 추가: 65이상 / 55 / 50혹은48 / 43혹은42 / 그외
    CASE 
        WHEN inch >= 65 THEN '65이상'
        WHEN inch = 55 THEN '55'
        WHEN inch IN (50, 48) THEN '50/48'
        WHEN inch IN (43, 42) THEN '43/42'
        ELSE '기타'
    END AS inch_cate,
     
    -- 시리즈 카테고리 컬럼 추가: M5, G5, C5, B5, QNED9M, Q93 등 지정된 값만
    CASE
        WHEN series = 'M5' THEN 'M5'
        WHEN series = 'G5' THEN 'G5'
        WHEN series = 'C5' THEN 'C5'
        WHEN series = 'B5' THEN 'B5'
        WHEN series = 'QNED9M' THEN 'QNED9M'
        WHEN series = 'QNED93' THEN 'QNED93'
        WHEN series = 'QNED92' THEN 'QNED92'
        WHEN series = 'QNED91' THEN 'QNED91'
        WHEN series = 'QNED90' THEN 'QNED90'
        WHEN series = 'QNED87' THEN 'QNED87'
        WHEN series = 'QNED86' THEN 'QNED86'
        WHEN series = 'QNED85' THEN 'QNED85'
        ELSE '기타'
    END AS series_cate,

    COUNT(1),
    COUNT(distinct mac_addr)
FROM 
    (
    SELECT 
        mac_addr,
        sales_model_code,
        -- 인치 추출: OLED 모델명 5~6번째, 그 외 기존 로직
        CASE 
            WHEN sales_model_code LIKE 'OLED%' AND REGEXP_EXTRACT(sales_model_code, '^OLED([0-9]{2})', 1) != '' 
                THEN CAST(REGEXP_EXTRACT(sales_model_code, '^OLED([0-9]{2})', 1) AS INT)
            WHEN REGEXP_EXTRACT(sales_model_code, '^([0-9]{2})', 1) != '' 
                THEN CAST(REGEXP_EXTRACT(sales_model_code, '^([0-9]{2})', 1) AS INT)
            WHEN sales_model_code LIKE 'QNED%' AND REGEXP_EXTRACT(sales_model_code, 'QNED([0-9]{2})', 1) != '' 
                THEN CAST(REGEXP_EXTRACT(sales_model_code, 'QNED([0-9]{2})', 1) AS INT)
            WHEN sales_model_code LIKE 'NANO%' AND REGEXP_EXTRACT(sales_model_code, 'NANO([0-9]{2})', 1) != '' 
                THEN CAST(REGEXP_EXTRACT(sales_model_code, 'NANO([0-9]{2})', 1) AS INT)
            WHEN sales_model_code LIKE 'MRGB%' AND REGEXP_EXTRACT(sales_model_code, 'MRGB([0-9]{2})', 1) != '' 
                THEN CAST(REGEXP_EXTRACT(sales_model_code, 'MRGB([0-9]{2})', 1) AS INT)
            WHEN sales_model_code LIKE 'LX%' AND REGEXP_EXTRACT(sales_model_code, 'LX([0-9]{2})', 1) != '' 
                THEN CAST(REGEXP_EXTRACT(sales_model_code, 'LX([0-9]{2})', 1) AS INT)
            WHEN sales_model_code LIKE 'UA%' AND REGEXP_EXTRACT(sales_model_code, 'UA([0-9]{2})', 1) != '' 
                THEN CAST(REGEXP_EXTRACT(sales_model_code, 'UA([0-9]{2})', 1) AS INT)
            ELSE NULL
        END AS inch,
        -- 시리즈 추출: OLED는 7~8번째(예: M5, G5, C5, B5 등), CS5 등 7~9번째(예외), QNED는 5~9번째
        CASE
            -- OLED: CS5 시리즈 (예외, 7~9번째)
            WHEN sales_model_code LIKE 'OLED%' AND REGEXP_EXTRACT(sales_model_code, '^OLED[0-9]{2}(CS5)', 1) != ''
                THEN REGEXP_EXTRACT(sales_model_code, '^OLED[0-9]{2}(CS5)', 1)
            -- OLED: 일반 시리즈 (7~8번째, M5/G5/C5/B5 등)
            WHEN sales_model_code LIKE 'OLED%' AND REGEXP_EXTRACT(sales_model_code, '^OLED[0-9]{2}([A-Z][0-9])', 1) != ''
                THEN REGEXP_EXTRACT(sales_model_code, '^OLED[0-9]{2}([A-Z][0-9])', 1)
            -- QNED: QNED9M, Q93 등
            WHEN sales_model_code LIKE 'QNED%' AND REGEXP_EXTRACT(sales_model_code, '^QNED([0-9]M)', 1) != ''
                THEN CONCAT('QNED', REGEXP_EXTRACT(sales_model_code, '^QNED([0-9]M)', 1))
            WHEN sales_model_code LIKE 'Q%' AND REGEXP_EXTRACT(sales_model_code, '^Q([0-9]{2})', 1) != ''
                THEN CONCAT('Q', REGEXP_EXTRACT(sales_model_code, '^Q([0-9]{2})', 1))
            ELSE NULL
        END AS series
    -- FROM aic_data_ods.tlamp.normal_log_webos25
    FROM aic_data_mart.tv_usage.hdmi_usage
    WHERE 1=1
        AND date_ym='2026-05'
        AND platform_code like 'W25%'
    )
WHERE 1=1
GROUP BY 
    inch_cate, series_cate
ORDER BY 
    inch_cate, series_cate
;

In [0]:
%sql
SELECT 
    date_ym,
    -- 인치 카테고리 컬럼 추가: 65이상 / 55 / 50혹은48 / 43혹은42 / 그외
    CASE 
        WHEN TRY_CAST(inch_size AS INT) >= 65 THEN '65이상'
        WHEN TRY_CAST(inch_size AS INT) = 55 THEN '55'
        WHEN TRY_CAST(inch_size AS INT) IN (50, 48) THEN '50/48'
        WHEN TRY_CAST(inch_size AS INT) IN (43, 42) THEN '43/42'
        ELSE '기타'
    END AS inch_cate,
     
    -- 시리즈 카테고리 컬럼 추가: M5, G5, C5, B5, QNED9M, Q93 등 지정된 값만
    CASE
        WHEN series_name = 'M5' THEN 'M5'
        WHEN series_name = 'G5' THEN 'G5'
        WHEN series_name = 'C5' THEN 'C5'
        WHEN series_name = 'B5' THEN 'B5'
        WHEN series_name = 'QNED9M' THEN 'QNED9M'
        WHEN series_name = 'QNED93' THEN 'QNED93'
        WHEN series_name = 'QNED92' THEN 'QNED92'
        WHEN series_name = 'QNED91' THEN 'QNED91'
        WHEN series_name = 'QNED90' THEN 'QNED90'
        WHEN series_name = 'QNED87' THEN 'QNED87'
        WHEN series_name = 'QNED86' THEN 'QNED86'
        WHEN series_name = 'QNED85' THEN 'QNED85'
        ELSE '기타'
    END AS series_cate,

    COUNT(1),
    COUNT(distinct mac_addr)
FROM 
    (
    SELECT
        date_ym, 
        mac_addr,
        country_code,
        sales_model_code,
        -- 제품군 구분
        CASE
            WHEN sales_model_code LIKE 'OLED%' THEN 'OLED'
            WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}QNED') THEN 'QNED'
            WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}NANO') THEN 'NANO'
            WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}UA') THEN 'UHD'
            ELSE 'UNKNOWN'
        END AS product_line,
        -- 시리즈명 추출
        CASE
            WHEN sales_model_code LIKE 'OLED%' 
                THEN REGEXP_EXTRACT(sales_model_code, '^OLED\[0-9]{2,3}([A-Z]{1,2}\[0-9])', 1)
            WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}QNED\[0-9]{2}')
                THEN REGEXP_EXTRACT(sales_model_code, '^\[0-9]{2,3}(QNED\[0-9]{2})', 1)
            WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}NANO\[0-9]{2}')
                THEN REGEXP_EXTRACT(sales_model_code, '^\[0-9]{2,3}(NANO\[0-9]{2})', 1)
            WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}UA\[0-9]{2}')
                THEN REGEXP_EXTRACT(sales_model_code, '^\[0-9]{2,3}(UA\[0-9]{2})', 1)
            ELSE NULL
        END AS series_name,
        -- 인치(사이즈) 추출
        CASE
            WHEN sales_model_code LIKE 'OLED%'
                THEN REGEXP_EXTRACT(sales_model_code, '^OLED(\[0-9]{2,3})', 1)
            WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}')
                THEN REGEXP_EXTRACT(sales_model_code, '^(\[0-9]{2,3})', 1)
            ELSE NULL
        END AS inch_size
    FROM aic_data_mart.tv_usage.hdmi_usage hu
    WHERE 1=1
        AND date_ym='2026-05'
        -- AND platform_code like 'W25%'
        AND platform_version='webOS25'
    )
WHERE 1=1
GROUP BY 
    date_ym, inch_cate, series_cate
ORDER BY 
    date_ym, inch_cate, series_cate
;

In [0]:
%sql
    SELECT
        date_ym, 
        mac_addr,
        country_code,
        sales_model_code,
        -- 제품군 구분
        CASE
            WHEN sales_model_code LIKE 'OLED%' THEN 'OLED'
            WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}QNED') THEN 'QNED'
            WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}NANO') THEN 'NANO'
            WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}UA') THEN 'UHD'
            ELSE 'UNKNOWN'
        END AS product_line,
        -- 시리즈명 추출
        CASE
            WHEN sales_model_code LIKE 'OLED%' 
                THEN REGEXP_EXTRACT(sales_model_code, '^OLED\[0-9]{2,3}([A-Z]{1,2}\[0-9])', 1)
            WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}QNED\[0-9]{2}')
                THEN REGEXP_EXTRACT(sales_model_code, '^\[0-9]{2,3}(QNED\[0-9]{2})', 1)
            WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}NANO\[0-9]{2}')
                THEN REGEXP_EXTRACT(sales_model_code, '^\[0-9]{2,3}(NANO\[0-9]{2})', 1)
            WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}UA\[0-9]{2}')
                THEN REGEXP_EXTRACT(sales_model_code, '^\[0-9]{2,3}(UA\[0-9]{2})', 1)
            ELSE NULL
        END AS series_name,
        -- 인치(사이즈) 추출
        CASE
            WHEN sales_model_code LIKE 'OLED%'
                THEN REGEXP_EXTRACT(sales_model_code, '^OLED(\[0-9]{2,3})', 1)
            WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}')
                THEN REGEXP_EXTRACT(sales_model_code, '^(\[0-9]{2,3})', 1)
            ELSE NULL
        END AS inch_size
    FROM aic_data_mart.tv_usage.hdmi_usage hu
    WHERE 1=1
        AND date_ym='2026-05'
        AND platform_version='webOS25'
        AND CASE
                WHEN sales_model_code LIKE 'OLED%' THEN 'OLED'
                WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}QNED') THEN 'QNED'
                WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}NANO') THEN 'NANO'
                WHEN REGEXP_LIKE(sales_model_code, '^\[0-9]{2,3}UA') THEN 'UHD'
                ELSE 'UNKNOWN'
            END in ('OLED','QNED')


In [0]:
%sql
-- 기기별 HDMI / PC / 게임 콘솔 연결 여부 플래그 (per mac_addr)
-- hdmi_info struct: hdmi_id(string), hdmi_type(string), hdmi_label(string), log_dt(timestamp)
-- hdmi_use_dict: HDMI_1~HDMI_4 boolean (포트별 사용여부)
SELECT
    date_ym,
    mac_addr,
    country_code,
    sales_model_code,
    platform_version,

    -- (2) HDMI 1~4 포트 연결 여부: hdmi_use_dict 기준
    CASE WHEN (hdmi_use_dict.HDMI_1 = true
            OR hdmi_use_dict.HDMI_2 = true
            OR hdmi_use_dict.HDMI_3 = true
            OR hdmi_use_dict.HDMI_4 = true)
         THEN 1 ELSE 0 END                                           AS hdmi_connected_flag,

    -- (3) PC 연결 여부: extInput 타입 + label에 'pc' 포함 (대소문자 무관)
    --     서버에서 hdmi_type='pc' 로 내려오는 케이스 없음 → extInput + label로 판별
    CASE WHEN EXISTS(hdmi_info, h -> h.hdmi_type = 'extInput'
                                 AND LOWER(h.hdmi_label) LIKE '%pc%')
         THEN 1 ELSE 0 END                                           AS pc_connected_flag,

    -- (4) 게임 콘솔 연결 여부: hdmi_type='game'
    CASE WHEN EXISTS(hdmi_info, h -> h.hdmi_type = 'game')
         THEN 1 ELSE 0 END                                           AS game_connected_flag

FROM aic_data_mart.tv_usage.hdmi_usage
WHERE 1=1
    AND date_ym = '2026-05'
    AND platform_version = 'webOS25'

In [0]:
%sql
-- Cell 8 dimension + Cell 9 HDMI/PC/게임 플래그 합친 집계
-- 같은 hdmi_usage 테이블에서 dimension 추출 + 플래그를 CTE 하나로 통합
WITH base AS (
    SELECT
        mac_addr,
        country_code,
        sales_model_code,

        -- 제품군 구분 (Cell 8 동일 로직)
        CASE
            WHEN sales_model_code LIKE 'OLED%'                        THEN 'OLED'
            WHEN REGEXP_LIKE(sales_model_code, '^[0-9]{2,3}QNED')       THEN 'QNED'
            WHEN REGEXP_LIKE(sales_model_code, '^[0-9]{2,3}NANO')       THEN 'NANO'
            WHEN REGEXP_LIKE(sales_model_code, '^[0-9]{2,3}UA')         THEN 'UHD'
            ELSE 'UNKNOWN'
        END AS product_line,

        -- 시리즈명 추출 (Cell 8 동일 로직)
        CASE
            WHEN sales_model_code LIKE 'OLED%'
                THEN REGEXP_EXTRACT(sales_model_code, '^OLED[0-9]{2,3}([A-Z]{1,2}[0-9])', 1)
            WHEN REGEXP_LIKE(sales_model_code, '^[0-9]{2,3}QNED[0-9]{2}')
                THEN REGEXP_EXTRACT(sales_model_code, '^[0-9]{2,3}(QNED[0-9]{2})', 1)
            WHEN REGEXP_LIKE(sales_model_code, '^[0-9]{2,3}NANO[0-9]{2}')
                THEN REGEXP_EXTRACT(sales_model_code, '^[0-9]{2,3}(NANO[0-9]{2})', 1)
            WHEN REGEXP_LIKE(sales_model_code, '^[0-9]{2,3}UA[0-9]{2}')
                THEN REGEXP_EXTRACT(sales_model_code, '^[0-9]{2,3}(UA[0-9]{2})', 1)
            ELSE NULL
        END AS series_name,

        -- 인치 추출 (Cell 8 동일 로직)
        CASE
            WHEN sales_model_code LIKE 'OLED%'
                THEN REGEXP_EXTRACT(sales_model_code, '^OLED([0-9]{2,3})', 1)
            WHEN REGEXP_LIKE(sales_model_code, '^[0-9]{2,3}')
                THEN REGEXP_EXTRACT(sales_model_code, '^([0-9]{2,3})', 1)
            ELSE NULL
        END AS inch_size,

        -- (2) HDMI 1~4 연결 여부 (Cell 9 동일 로직)
        CASE WHEN (hdmi_use_dict.HDMI_1 = true
                OR hdmi_use_dict.HDMI_2 = true
                OR hdmi_use_dict.HDMI_3 = true
                OR hdmi_use_dict.HDMI_4 = true)
             THEN 1 ELSE 0 END AS hdmi_connected_flag,

        -- (3) PC 연결 여부 (Cell 9 동일 로직)
        CASE WHEN EXISTS(hdmi_info, h -> h.hdmi_type = 'extInput'
                                     AND LOWER(h.hdmi_label) LIKE '%pc%')
             THEN 1 ELSE 0 END AS pc_connected_flag,

        -- (4) 게임 콘솔 연결 여부 (Cell 9 동일 로직)
        CASE WHEN EXISTS(hdmi_info, h -> h.hdmi_type = 'game')
             THEN 1 ELSE 0 END AS game_connected_flag

    FROM aic_data_mart.tv_usage.hdmi_usage
    WHERE 1=1
        AND date_ym        = '2026-05'
        AND platform_version = 'webOS25'
)
SELECT
    -- inch 카테고리 (Cell 7 동일 로직)
    CASE
        WHEN TRY_CAST(inch_size AS INT) >= 65            THEN '65이상'
        WHEN TRY_CAST(inch_size AS INT) = 55             THEN '55'
        WHEN TRY_CAST(inch_size AS INT) IN (50, 48)      THEN '50/48'
        WHEN TRY_CAST(inch_size AS INT) IN (43, 42)      THEN '43/42'
        ELSE '기타'
    END AS inch_cate,

    -- 시리즈 카테고리 (Cell 7 동일 로직)
    CASE
        WHEN series_name IN ('M5','G5','C5','B5')                     THEN series_name
        WHEN series_name IN ('QNED9M','QNED93','QNED92','QNED91',
                             'QNED90','QNED87','QNED86','QNED85')     THEN series_name
        ELSE '기타'
    END AS series_cate,

    -- 집계 카운트
    COUNT(DISTINCT mac_addr)                                           AS total_tv,
    COUNT(DISTINCT CASE WHEN hdmi_connected_flag = 1 THEN mac_addr END) AS hdmi_connected_tv,
    COUNT(DISTINCT CASE WHEN pc_connected_flag   = 1 THEN mac_addr END) AS pc_connected_tv,
    COUNT(DISTINCT CASE WHEN game_connected_flag = 1 THEN mac_addr END) AS game_connected_tv

FROM base
WHERE product_line IN ('OLED', 'QNED')   -- Cell 8 필터 동일 적용
GROUP BY inch_cate, series_cate
ORDER BY inch_cate, series_cate
;

In [0]:
%sql
WITH raw AS (
    SELECT
        date_ym, 
        mac_addr,
        country_code,
        sales_model_code,
        CASE
            WHEN sales_model_code LIKE 'OLED%' THEN 'OLED'
            WHEN REGEXP_LIKE(sales_model_code, '^\\d{2,3}QNED') THEN 'QNED'
            WHEN REGEXP_LIKE(sales_model_code, '^\\d{2,3}NANO') THEN 'NANO'
            WHEN REGEXP_LIKE(sales_model_code, '^\\d{2,3}UA') THEN 'UHD'
            ELSE 'UNKNOWN'
        END AS product_line,
        CASE
            WHEN sales_model_code LIKE 'OLED%' 
                THEN REGEXP_EXTRACT(sales_model_code, '^OLED\\d{2,3}([A-Z]{1,2}\\d)', 1)
            WHEN REGEXP_LIKE(sales_model_code, '^\\d{2,3}QNED\\d{2}')
                THEN REGEXP_EXTRACT(sales_model_code, '^\\d{2,3}(QNED\\d{2})', 1)
            WHEN REGEXP_LIKE(sales_model_code, '^\\d{2,3}NANO\\d{2}')
                THEN REGEXP_EXTRACT(sales_model_code, '^\\d{2,3}(NANO\\d{2})', 1)
            WHEN REGEXP_LIKE(sales_model_code, '^\\d{2,3}UA\\d{2}')
                THEN REGEXP_EXTRACT(sales_model_code, '^\\d{2,3}(UA\\d{2})', 1)
            ELSE 'UNKNOWN'
        END AS series_name,
        CASE
            WHEN sales_model_code LIKE 'OLED%'
                THEN REGEXP_EXTRACT(sales_model_code, '^OLED(\\d{2,3})', 1)
            WHEN REGEXP_LIKE(sales_model_code, '^\\d{2,3}')
                THEN REGEXP_EXTRACT(sales_model_code, '^(\\d{2,3})', 1)
            ELSE NULL
        END AS inch_size,
        CASE WHEN (hdmi_use_dict.HDMI_1 = true
                OR hdmi_use_dict.HDMI_2 = true
                OR hdmi_use_dict.HDMI_3 = true
                OR hdmi_use_dict.HDMI_4 = true)
            THEN 1 ELSE 0 END AS hdmi_connected_flag,
        CASE WHEN EXISTS(hdmi_info, h -> h.hdmi_type = 'extInput'
                                    AND LOWER(h.hdmi_label) LIKE '%pc%')
            THEN 1 ELSE 0 END AS pc_connected_flag,
        CASE WHEN EXISTS(hdmi_info, h -> h.hdmi_type = 'game')
            THEN 1 ELSE 0 END AS game_connected_flag
    FROM aic_data_mart.tv_usage.hdmi_usage
    WHERE date_ym BETWEEN '2025-04' and '2026-03' -- date_ym = '2026-05'
      AND platform_version = 'webOS25'
      AND CASE
              WHEN sales_model_code LIKE 'OLED%' THEN 'OLED'
              WHEN REGEXP_LIKE(sales_model_code, '^\\d{2,3}QNED') THEN 'QNED'
              WHEN REGEXP_LIKE(sales_model_code, '^\\d{2,3}NANO') THEN 'NANO'
              WHEN REGEXP_LIKE(sales_model_code, '^\\d{2,3}UA') THEN 'UHD'
              ELSE 'UNKNOWN'
          END IN ('OLED', 'QNED')
)

SELECT
    date_ym, 
    product_line,
    series_cate,
    inch_cate,
    COUNT(*)                        AS total_devices,
    SUM(hdmi_connected_flag)        AS hdmi_connected_cnt,
    SUM(pc_connected_flag)          AS pc_connected_cnt,
    SUM(game_connected_flag)        AS game_connected_cnt,
    ROUND(SUM(hdmi_connected_flag) * 100.0 / COUNT(*), 1) AS hdmi_connected_pct,
    ROUND(SUM(pc_connected_flag)   * 100.0 / COUNT(*), 1) AS pc_connected_pct,
    ROUND(SUM(game_connected_flag) * 100.0 / COUNT(*), 1) AS game_connected_pct
FROM (
    SELECT
        *,
        CASE 
            WHEN TRY_CAST(inch_size AS INT) >= 65 THEN '65이상'
            WHEN TRY_CAST(inch_size AS INT) = 55   THEN '55'
            WHEN TRY_CAST(inch_size AS INT) IN (50, 48) THEN '50/48'
            WHEN TRY_CAST(inch_size AS INT) IN (43, 42) THEN '43/42'
            ELSE '기타'
        END AS inch_cate,
        CASE
            WHEN series_name = 'M5'     THEN 'M5'
            WHEN series_name = 'G5'     THEN 'G5'
            WHEN series_name = 'C5'     THEN 'C5'
            WHEN series_name = 'B5'     THEN 'B5'
            WHEN series_name = 'QNED9M' THEN 'QNED9M'
            WHEN series_name = 'QNED93' THEN 'QNED93'
            WHEN series_name = 'QNED92' THEN 'QNED92'
            WHEN series_name = 'QNED91' THEN 'QNED91'
            WHEN series_name = 'QNED90' THEN 'QNED90'
            WHEN series_name = 'QNED87' THEN 'QNED87'
            WHEN series_name = 'QNED86' THEN 'QNED86'
            WHEN series_name = 'QNED85' THEN 'QNED85'
            ELSE '기타'
        END AS series_cate
    FROM raw
) t
GROUP BY date_ym, product_line, series_cate, inch_cate
ORDER BY date_ym, product_line, series_cate, inch_cate

In [0]:
%sql
-- ① 제품군
CREATE OR REPLACE FUNCTION sandbox.z_yeswook_kim.fn_product_line(s STRING)
RETURNS STRING
RETURN CASE
    WHEN s LIKE 'OLED%'                        THEN 'OLED'
    WHEN REGEXP_LIKE(s, '^\\d{2,3}QNED')      THEN 'QNED'
    WHEN REGEXP_LIKE(s, '^\\d{2,3}NANO')      THEN 'NANO'
    WHEN REGEXP_LIKE(s, '^\\d{2,3}UA')        THEN 'UHD'
    ELSE 'UNKNOWN'
END;

-- ② 시리즈명
CREATE OR REPLACE FUNCTION sandbox.z_yeswook_kim.fn_series_name(s STRING)
RETURNS STRING
RETURN CASE
    WHEN s LIKE 'OLED%'
        THEN REGEXP_EXTRACT(s, '^OLED\\d{2,3}([A-Z]{1,2}\\d)', 1)
    WHEN REGEXP_LIKE(s, '^\\d{2,3}QNED\\d{2}')
        THEN REGEXP_EXTRACT(s, '^\\d{2,3}(QNED\\d{2})', 1)
    WHEN REGEXP_LIKE(s, '^\\d{2,3}NANO\\d{2}')
        THEN REGEXP_EXTRACT(s, '^\\d{2,3}(NANO\\d{2})', 1)
    WHEN REGEXP_LIKE(s, '^\\d{2,3}UA\\d{2}')
        THEN REGEXP_EXTRACT(s, '^\\d{2,3}(UA\\d{2})', 1)
    ELSE 'UNKNOWN'
END;

-- ③ 인치 추출
CREATE OR REPLACE FUNCTION sandbox.z_yeswook_kim.fn_inch_size(s STRING)
RETURNS STRING
RETURN CASE
    WHEN s LIKE 'OLED%'
        THEN REGEXP_EXTRACT(s, '^OLED(\\d{2,3})', 1)
    WHEN REGEXP_LIKE(s, '^\\d{2,3}')
        THEN REGEXP_EXTRACT(s, '^(\\d{2,3})', 1)
    ELSE NULL
END;

-- ④ 인치 카테고리
CREATE OR REPLACE FUNCTION sandbox.z_yeswook_kim.fn_inch_cate(inch STRING)
RETURNS STRING
RETURN CASE
    WHEN TRY_CAST(inch AS INT) >= 65         THEN '65이상'
    WHEN TRY_CAST(inch AS INT) = 55          THEN '55'
    WHEN TRY_CAST(inch AS INT) IN (50, 48)   THEN '50/48'
    WHEN TRY_CAST(inch AS INT) IN (43, 42)   THEN '43/42'
    ELSE '기타'
END;

-- ⑤ 시리즈 카테고리 (1423 전용 → sandbox.z_yeswook_kim)
CREATE OR REPLACE FUNCTION sandbox.z_yeswook_kim.fn_series_cate(sname STRING)
RETURNS STRING
RETURN CASE
    WHEN sname = 'M5'     THEN 'M5'
    WHEN sname = 'G5'     THEN 'G5'
    WHEN sname = 'C5'     THEN 'C5'
    WHEN sname = 'B5'     THEN 'B5'
    WHEN sname = 'QNED9M' THEN 'QNED9M'
    WHEN sname = 'QNED93' THEN 'QNED93'
    WHEN sname = 'QNED92' THEN 'QNED92'
    WHEN sname = 'QNED91' THEN 'QNED91'
    WHEN sname = 'QNED90' THEN 'QNED90'
    WHEN sname = 'QNED87' THEN 'QNED87'
    WHEN sname = 'QNED86' THEN 'QNED86'
    WHEN sname = 'QNED85' THEN 'QNED85'
    ELSE '기타'
END

In [0]:
%sql
WITH base AS (
    SELECT
        date_ym,
        country_code,
        platform_code,
        sales_model_code,
        one_day_plus_device,
        one_day_plus_user,
        sandbox.z_yeswook_kim.fn_product_line(sales_model_code)  AS product_line,
        sandbox.z_yeswook_kim.fn_series_name(sales_model_code)   AS series_name,
        sandbox.z_yeswook_kim.fn_inch_size(sales_model_code)     AS inch_size
    FROM aic_data_fact.common_tv.tv_usage_monthly
    WHERE date_ym BETWEEN '2025-04' AND '2026-03'
      AND platform_code LIKE 'W25%'
      AND sandbox.z_yeswook_kim.fn_product_line(sales_model_code) IN ('OLED', 'QNED')
)

SELECT
    date_ym,
    product_line,
    sandbox.z_yeswook_kim.fn_series_cate(series_name)  AS series_cate,
    sandbox.z_yeswook_kim.fn_inch_cate(inch_size)      AS inch_cate,
    SUM(one_day_plus_device)                             AS total_device
    -- ,SUM(one_day_plus_user)                               AS total_user
FROM base
GROUP BY
    date_ym,
    product_line,
    sandbox.z_yeswook_kim.fn_series_cate(series_name),
    sandbox.z_yeswook_kim.fn_inch_cate(inch_size)
ORDER BY date_ym, product_line, series_cate, inch_cate


In [0]:
%sql
-- 집계단위: date_ym + sales_model_code + device_type + device_label
-- 지표: device_cnt (game/PC 기기별 연결 TV 대수)

WITH
base AS (
    SELECT
        date_ym,
        mac_addr,
        SPLIT_PART(sales_model_code, '.', 1) AS sales_model_code,
        hdmi_info
    FROM aic_data_mart.tv_usage.hdmi_usage
    WHERE date_ym        = '2026-03'
      AND platform_version = 'webOS25'
),

device_raw AS (
    SELECT
        date_ym,
        mac_addr,
        sales_model_code,
        CASE
            WHEN h.hdmi_type = 'game'                                           THEN 'game'
            WHEN h.hdmi_type = 'extInput' AND LOWER(h.hdmi_label) LIKE '%pc%'  THEN 'PC'
        END AS device_type,
        h.hdmi_label AS device_label
    FROM base
    LATERAL VIEW EXPLODE(hdmi_info) AS h
    WHERE    h.hdmi_type = 'game'
          OR (h.hdmi_type = 'extInput' AND LOWER(h.hdmi_label) LIKE '%pc%')
)

SELECT
    date_ym,
    sales_model_code,
    COALESCE(NULLIF(sandbox.z_yeswook_kim.fn_product_line(sales_model_code), 'UNKNOWN'), 'etc.') AS product,
    COALESCE(NULLIF(sandbox.z_yeswook_kim.fn_series_name(sales_model_code),  'UNKNOWN'), 'etc.') AS series,
    sandbox.z_yeswook_kim.fn_inch_size(sales_model_code)                                         AS inch,
    device_type,
    device_label,
    COUNT(DISTINCT mac_addr) AS device_cnt
FROM device_raw
GROUP BY date_ym, sales_model_code, device_type, device_label
ORDER BY sales_model_code, device_type, device_label

In [0]:
%sql
-- 집계단위: date_ym + product + series + inch
-- 지표: hdmi_connected_cnt (HDMI 1~4 사용 TV), total_connected_cnt (앱 사용 TV)

WITH
-- hdmi_usage: '.' 이후 제거 + 모델 차원 추출
hdmi_model AS (
    SELECT
        date_ym,
        mac_addr,
        SPLIT_PART(sales_model_code, '.', 1)                                                         AS sales_model_code,
        COALESCE(NULLIF(sandbox.z_yeswook_kim.fn_product_line(SPLIT_PART(sales_model_code, '.', 1)), 'UNKNOWN'), 'etc.') AS product,
        COALESCE(NULLIF(sandbox.z_yeswook_kim.fn_series_name(SPLIT_PART(sales_model_code, '.', 1)),  'UNKNOWN'), 'etc.') AS series,
        sandbox.z_yeswook_kim.fn_inch_size(SPLIT_PART(sales_model_code, '.', 1))                     AS inch,
        hdmi_use_dict
    FROM aic_data_mart.tv_usage.hdmi_usage
    WHERE date_ym        = '2026-03'
      AND platform_version = 'webOS25'
),

-- ① HDMI 1~4 아무 포트 사용 TV → sales_model_code + product/series/inch 단위 집계
hdmi_cnt AS (
    SELECT
        date_ym, sales_model_code, product, series, inch,
        COUNT(DISTINCT mac_addr) AS hdmi_connected_cnt
    FROM hdmi_model
    WHERE (   hdmi_use_dict.HDMI_1 = true
           OR hdmi_use_dict.HDMI_2 = true
           OR hdmi_use_dict.HDMI_3 = true
           OR hdmi_use_dict.HDMI_4 = true)
    GROUP BY date_ym, sales_model_code, product, series, inch
),

-- ② 앱 사용 이력 TV (tv_usage_monthly) → sales_model_code + product/series/inch 단위 집계
total_cnt AS (
    SELECT
        date_ym,
        SPLIT_PART(sales_model_code, '.', 1)                                                         AS sales_model_code,
        COALESCE(NULLIF(sandbox.z_yeswook_kim.fn_product_line(SPLIT_PART(sales_model_code, '.', 1)), 'UNKNOWN'), 'etc.') AS product,
        COALESCE(NULLIF(sandbox.z_yeswook_kim.fn_series_name(SPLIT_PART(sales_model_code, '.', 1)),  'UNKNOWN'), 'etc.') AS series,
        sandbox.z_yeswook_kim.fn_inch_size(SPLIT_PART(sales_model_code, '.', 1))                     AS inch,
        SUM(one_day_plus_device) AS total_connected_cnt
    FROM aic_data_fact.common_tv.tv_usage_monthly
    WHERE date_ym       = '2026-03'
      AND platform_code LIKE 'W25%'
    GROUP BY date_ym, SPLIT_PART(sales_model_code, '.', 1), product, series, inch
)

SELECT
    h.date_ym,
    h.sales_model_code,
    h.product,
    h.series,
    h.inch,
    h.hdmi_connected_cnt,
    t.total_connected_cnt
FROM hdmi_cnt h
LEFT JOIN total_cnt t
    ON  h.date_ym          = t.date_ym
    AND h.sales_model_code = t.sales_model_code
ORDER BY h.date_ym, h.sales_model_code